# Superpixel Explorer

Segmentation-free **superpixel** feature tables for multiplex tissue imaging.

The image is divided into a regular grid of square regions ("superpixels") and
per-region marker statistics (mean / sum / std) are aggregated into a feature
table that plugs straight into the downstream `analysis` pipeline
(preprocessing → clustering → visualization → GeoJSON export).

Use the interactive plot below to tune the square size by dragging the slider,
optionally drop empty (background / low-signal) superpixels with a threshold,
then export the feature table. A batch cell at the end runs the same extraction
over every MCMICRO ROI.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path if running from the notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from pseudochannel import OMETiffChannels
from analysis.superpixels import (
    extract_superpixel_features,
    compute_superpixel_features,
    compute_superpixel_features_batch,
)
from analysis.superpixel_widgets import create_superpixel_explorer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

%matplotlib widget

## Configuration

Point `IMAGE_PATH` at one backsub OME-TIFF and `MARKER_FILE` at its
`markers.csv`. All channels in the marker file become feature columns (no
default exclusion).

In [ ]:
# --- Single image to tune on ---
IMAGE_PATH = "../background/image.ome.tiff"   # <-- edit
MARKER_FILE = "../markers.csv"                # <-- edit
MCMICRO_MARKERS = True                        # markers.csv is MCMICRO format

## Load Channels

In [ ]:
channels = OMETiffChannels(
    IMAGE_PATH,
    MARKER_FILE,
    exclude_channels=set(),      # keep ALL channels as features
    mcmicro_markers=MCMICRO_MARKERS,
)
print(f"Image shape: {channels.shape}")
print(f"{len(channels.marker_names)} channels: {channels.marker_names}")

## Interactive Superpixel Tuning

Drag **Size (px)** to change the square side length; the cyan grid re-tiles
live. Tick **Remove empty superpixels** and drag **Empty pctl** to set an
adaptive cutoff: superpixels at or below that percentile of total signal are
dropped (e.g. 25 removes the lowest-signal ~25%). Removed blocks simply lose
their grid outline, so the grid visibly dissolves over background while the kept
(tissue) superpixels stay outlined. Use **Show** to switch the background
between a max projection and any single channel. Click **Compute table** (or call
`explorer.export_features(...)` below) to run the full-resolution extraction with
the current settings.

In [ ]:
explorer = create_superpixel_explorer(channels, initial_size=64)

## Save Superpixel Masks (Cellpose + MacsIqView)

First output step — save the superpixel grid as masks in both downstream
formats, using the size and threshold you tuned above:

- **Cellpose format** (`superpixel_cellpose.tif`): a label mask, one unique
  label per superpixel (matches the feature table's `label` column).
- **MacsIqView format** (`superpixel_MacsIQView.tif`): the binary separated mask
  from `Separate_masks.py` (1px gaps between regions).

When **Remove empty superpixels** is ticked, both masks are thresholded (empty
superpixels set to background) so they stay consistent with the feature table.

In [ ]:
explorer.export_features(
    save_masks=True,
    mask_dir="../analysis",
    mask_basename="superpixel",
    mask_formats=("cellpose", "macsiqview"),
)
print("Saved masks:", explorer.mask_paths)

## Export Feature Table

Runs the full-resolution extraction with the current slider settings. The
returned table has metadata columns (`label, tile_row, tile_col, centroid_y,
centroid_x, n_pixels, total_signal`) plus `{marker}_mean/_sum/_std` per channel.

In [ ]:
df = explorer.export_features(
    output_path="../analysis/superpixel_features.csv",  # optional; set None to skip
    save_labels=None,                                    # e.g. "../analysis/superpixel_labels.tif"
)
print(f"{len(df)} superpixels x {df.shape[1]} columns")
df.head()

## (Optional) Headless single-image extraction

Same result without the widget — useful for scripting a chosen size/threshold.

In [ ]:
df_headless = compute_superpixel_features(
    IMAGE_PATH,
    MARKER_FILE,
    size=64,
    mcmicro_markers=MCMICRO_MARKERS,
    remove_empty=True,
    empty_percentile=25,   # drop the lowest-signal ~25% of superpixels
    output_path=None,
)
df_headless.head()

## Batch Over ROIs

Runs superpixel extraction across every MCMICRO experiment under `ROOT_DIR`
(each `background/` folder with a sibling `markers.csv`). Writes a
`superpixel_features.csv` per ROI under `<rack>/analysis/` and returns the
concatenated table with an `ROI` column.

In [ ]:
ROOT_DIR = "../"           # <-- study root containing experiment folders
CHOSEN_SIZE = 64           # pick the size you tuned above
CHOSEN_PERCENTILE = 25     # empty-removal percentile (lowest-signal % to drop)

combined = compute_superpixel_features_batch(
    ROOT_DIR,
    size=CHOSEN_SIZE,
    remove_empty=True,
    empty_percentile=CHOSEN_PERCENTILE,
    save_masks=True,        # also write Cellpose + MacsIqView masks per ROI
    mask_formats=("cellpose", "macsiqview"),
    overwrite=False,
)
print(f"Combined: {len(combined)} superpixels across "
      f"{combined['ROI'].nunique() if len(combined) else 0} ROIs")
combined.head()